In [46]:
from transformers import AutoTokenizer, AutoConfig, EncoderDecoderModel, PretrainedConfig
import torch
import torch.nn as nn
import torch.nn.functional as F
from math import sqrt
from typing import Optional

## Decoder

This notebook builds a decoder-only transformer block **from scratch** with plain `torch.nn` modules, mirroring the architecture used by the decoder half of `patrickvonplaten/bert2bert_cnn_daily_mail` (loaded below only to reuse its `config`, tokenizer, and hidden sizes).

The build proceeds bottom-up:
1. Manually compute masked scaled dot-product attention on raw embeddings, to see each step explicitly.
2. Wrap that logic into a reusable `scaled_dot_product_attention` function.
3. Compose it into `AttentionHead` → `MultiHeadAttention`.
4. Add a `FeedForward` block and combine everything into a pre-norm `TransformerDecoderLayer`.
5. Add learned token + positional `Embeddings`.
6. Stack everything into a full `TransformerDecoder`.

In [47]:
model_ckpt = "patrickvonplaten/bert2bert_cnn_daily_mail"
text = "time flies like an arrow"

tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model = EncoderDecoderModel.from_pretrained(model_ckpt)

Config of the encoder: <class 'transformers.models.bert.modeling_bert.BertModel'> is overwritten by shared encoder config: BertConfig {
  "_name_or_path": "bert-base-uncased",
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "return_dict": false,
  "transformers_version": "4.46.3",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

Config of the decoder: <class 'transformers.models.bert.modeling_bert.BertLMHeadModel'> is overwritten by shared decoder config: BertConfig {
  "_name_or_path": "bert-base-uncased",
  "add_cross_attention"

In [48]:
model

EncoderDecoderModel(
  (encoder): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), ep

In [49]:
inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False)
inputs.input_ids

tensor([[ 2051, 10029,  2066,  2019,  8612]])

In [50]:
config = AutoConfig.from_pretrained(model_ckpt)

In [51]:
token_emb = nn.Embedding(config.vocab_size, config.decoder.hidden_size)
token_emb

Embedding(30522, 768)

In [52]:
inputs_embeds = token_emb(inputs.input_ids)
inputs_embeds.size()

torch.Size([1, 5, 768])

### Manual masked attention

Before wrapping this in a function, we compute causal (masked) self-attention step by step on the raw token embeddings:

- Build a **causal mask** (lower-triangular) so each position can only attend to itself and earlier positions — required for an autoregressive decoder.
- Compute raw attention **scores** as scaled dot products between queries and keys.
- Apply the mask by setting disallowed positions to `-inf` before the softmax, so they get zero weight.
- Turn the masked scores into **weights** with softmax, then take the weighted sum of values.

In [53]:
seq_len = inputs.input_ids.size(-1)
mask = torch.tril(torch.ones(seq_len, seq_len)).unsqueeze(0)
mask[0]

tensor([[1., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0.],
        [1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1.]])

In [54]:
query = key = value = inputs_embeds
dim_k = key.size(-1)
scores = torch.bmm(query, key.transpose(1, 2)) / sqrt(dim_k)
scores.size()

torch.Size([1, 5, 5])

In [55]:
scores.masked_fill(mask == 0, -float("inf"))

tensor([[[27.9271,    -inf,    -inf,    -inf,    -inf],
         [ 1.1296, 27.8898,    -inf,    -inf,    -inf],
         [-0.9311, -2.4570, 27.7286,    -inf,    -inf],
         [-0.3098,  0.9539,  0.5720, 27.5930,    -inf],
         [ 1.3380,  0.2574, -0.2069, -1.7805, 29.9447]]],
       grad_fn=<MaskedFillBackward0>)

In [56]:
weights = F.softmax(scores, dim=-1)

In [57]:
attn_outputs = torch.bmm(weights, value)
attn_outputs.shape

torch.Size([1, 5, 768])

### Reusable attention function

The manual steps above are now packaged into `scaled_dot_product_attention`, so the rest of the notebook can call it instead of repeating the score/mask/softmax logic every time.

In [58]:
def scaled_dot_product_attention(
    query: torch.Tensor,
    key: torch.Tensor,
    value: torch.Tensor,
    mask: Optional[torch.Tensor] = None,
) -> torch.Tensor:
    """Compute scaled dot-product attention.

    Args:
        query: Query tensor of shape (batch_size, seq_len, head_dim).
        key: Key tensor of shape (batch_size, seq_len, head_dim).
        value: Value tensor of shape (batch_size, seq_len, head_dim).
        mask: Optional mask broadcastable to the attention scores; positions
            where mask == 0 are set to -inf before the softmax.

    Returns:
        Attention output tensor of shape (batch_size, seq_len, head_dim).
    """
    dim_k = query.size(-1)
    scores = torch.bmm(query, key.transpose(1, 2)) / sqrt(dim_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -float('inf'))
    weights = F.softmax(scores, dim=-1)
    return torch.bmm(weights, value)

### Attention head

`AttentionHead` adds the missing piece from the manual demo above: **learned** linear projections (`q`, `k`, `v`) applied to the hidden state before attention, instead of using the raw embeddings directly as query/key/value.

In [59]:
class AttentionHead(nn.Module):
    """A single scaled dot-product attention head."""

    def __init__(self, embed_dim: int, head_dim: int) -> None:
        """Initialize the query, key, and value projections.

        Args:
            embed_dim: Dimensionality of the input embeddings.
            head_dim: Dimensionality of this head's projected space.
        """
        super().__init__()
        self.q = nn.Linear(embed_dim, head_dim)
        self.k = nn.Linear(embed_dim, head_dim)
        self.v = nn.Linear(embed_dim, head_dim)

    def forward(self, hidden_state: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        """Project the input and apply scaled dot-product attention.

        Args:
            hidden_state: Input tensor of shape (batch_size, seq_len, embed_dim).
            mask: Optional attention mask passed through to scaled_dot_product_attention.

        Returns:
            Attention output tensor of shape (batch_size, seq_len, head_dim).
        """
        return scaled_dot_product_attention(
            self.q(hidden_state), self.k(hidden_state), self.v(hidden_state), mask=mask
        )

### Multi-head attention

Instead of a single attention head, `MultiHeadAttention` runs `num_attention_heads` heads in parallel (each with `head_dim = embed_dim / num_heads`), concatenates their outputs, and projects the result back to `embed_dim` with `output_linear`. This lets the model attend to different representation subspaces at once.

In [60]:
class MultiHeadAttention(nn.Module):
    """Multi-head self-attention block combining several AttentionHead outputs."""

    def __init__(self, config: PretrainedConfig) -> None:
        """Build the attention heads and output projection.

        Args:
            config: Model config exposing decoder.hidden_size and
                decoder.num_attention_heads.
        """
        super().__init__()
        embed_dim = config.decoder.hidden_size
        num_heads = config.decoder.num_attention_heads
        head_dim = embed_dim // num_heads
        self.heads = nn.ModuleList(
            [AttentionHead(embed_dim, head_dim) for _ in range(num_heads)]
        )
        self.output_linear = nn.Linear(embed_dim, embed_dim)

    def forward(self, hidden_state: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        """Run all attention heads in parallel and merge their outputs.

        Args:
            hidden_state: Input tensor of shape (batch_size, seq_len, embed_dim).
            mask: Optional attention mask passed through to each head.

        Returns:
            Output tensor of shape (batch_size, seq_len, embed_dim).
        """
        x = torch.cat([h(hidden_state, mask=mask) for h in self.heads], dim = -1)
        return self.output_linear(x)

In [61]:
multihead_attention = MultiHeadAttention(config)
attn_outputs = multihead_attention(inputs_embeds, mask)
attn_outputs.shape

torch.Size([1, 5, 768])

### Feed-forward block

`FeedForward` is the position-wise sublayer applied independently to each token: expand `hidden_size` → `intermediate_size` with a GELU nonlinearity, then project back down to `hidden_size`, with dropout for regularization.

In [62]:
class FeedForward(nn.Module):
    """Position-wise feed-forward network used inside a transformer layer."""

    def __init__(self, config: PretrainedConfig) -> None:
        """Build the two linear layers, activation, and dropout.

        Args:
            config: Model config exposing decoder.hidden_size,
                decoder.intermediate_size, and decoder.hidden_dropout_prob.
        """
        super().__init__()
        self.linear_1 = nn.Linear(config.decoder.hidden_size, config.decoder.intermediate_size)
        self.linear_2 = nn.Linear(config.decoder.intermediate_size, config.decoder.hidden_size)
        self.gelu = nn.GELU()
        self.dropout = nn.Dropout(config.decoder.hidden_dropout_prob)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Apply the two-layer feed-forward transform.

        Args:
            x: Input tensor of shape (batch_size, seq_len, hidden_size).

        Returns:
            Output tensor of the same shape as the input.
        """
        x = self.linear_1(x)
        x = self.gelu(x)
        x = self.linear_2(x)
        x = self.dropout(x)
        return x

### Decoder layer

`TransformerDecoderLayer` combines `MultiHeadAttention` and `FeedForward` into a **pre-norm** transformer block: `LayerNorm` is applied *before* each sublayer, and the sublayer's output is added back via a residual connection (`x = x + sublayer(norm(x))`).

In [63]:
class TransformerDecoderLayer(nn.Module):
    """A single pre-norm transformer decoder layer (self-attention + feed-forward)."""

    def __init__(self, config: PretrainedConfig) -> None:
        """Build the layer norms, attention block, and feed-forward block.

        Args:
            config: Model config exposing decoder.hidden_size and the
                sub-config fields required by MultiHeadAttention and FeedForward.
        """
        super().__init__()
        self.layer_norm_1 = nn.LayerNorm(config.decoder.hidden_size)
        self.layer_norm_2 = nn.LayerNorm(config.decoder.hidden_size)
        self.attention = MultiHeadAttention(config)
        self.feed_forward = FeedForward(config)

    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        """Apply self-attention and feed-forward sublayers with residual connections.

        Args:
            x: Input tensor of shape (batch_size, seq_len, hidden_size).
            mask: Optional attention mask passed to the self-attention block.

        Returns:
            Output tensor of the same shape as the input.
        """
        hidden_state = self.layer_norm_1(x)
        x = x + self.attention(hidden_state, mask)
        x = x + self.feed_forward(self.layer_norm_2(x))
        return x

### Embeddings

`Embeddings` turns input token ids into vectors by summing a learned **token embedding** with a learned **positional embedding** (based on each token's index in the sequence), then applies `LayerNorm` and dropout. This gives the model both content and order information, since attention itself is permutation-invariant.

In [64]:
class Embeddings(nn.Module):
    """Token + learned positional embeddings with layer norm and dropout."""

    def __init__(self, config: PretrainedConfig) -> None:
        """Build the token and position embedding tables.

        Args:
            config: Model config exposing vocab_size and
                decoder.hidden_size, decoder.max_position_embeddings.
        """
        super().__init__()
        self.token_embeddings = nn.Embedding(config.vocab_size, config.decoder.hidden_size)
        self.position_embeddings = nn.Embedding(config.decoder.max_position_embeddings, config.decoder.hidden_size)
        self.layer_norm = nn.LayerNorm(config.decoder.hidden_size, eps=1e-12)
        self.dropout = nn.Dropout()

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        """Embed input token ids and add positional information.

        Args:
            input_ids: Tensor of token ids with shape (batch_size, seq_len).

        Returns:
            Embedding tensor of shape (batch_size, seq_len, hidden_size).
        """
        # Create position ids for input sequence
        seq_lenght = input_ids.size(1)
        position_ids = torch.arange(seq_lenght, dtype=torch.long,).unsqueeze(0)

        # Create token and position embeddings
        token_embeddings = self.token_embeddings(input_ids)
        position_embeddings = self.position_embeddings(position_ids)

        # Combinate token and position embeddings
        embeddings = token_embeddings + position_embeddings
        embeddings = self.layer_norm(embeddings)
        embeddings = self.dropout(embeddings)
        return embeddings

### Full decoder stack

`TransformerDecoder` puts everything together: it embeds the input ids, then runs them through `config.decoder.num_hidden_layers` stacked `TransformerDecoderLayer`s.

Note the causal mask is only passed to `self.layers[0]` — later layers are called without a mask. This is a deliberate simplification of this exercise (see the inline comment).

In [65]:
class TransformerDecoder(nn.Module):
    """Stack of transformer decoder layers on top of token + position embeddings."""

    def __init__(self, config: PretrainedConfig) -> None:
        """Build the embedding layer and the stack of decoder layers.

        Args:
            config: Model config exposing decoder.num_hidden_layers plus the
                fields required by Embeddings and TransformerDecoderLayer.
        """
        super().__init__()
        self.embeddings = Embeddings(config)
        self.layers = nn.ModuleList([TransformerDecoderLayer(config) for _ in range(config.decoder.num_hidden_layers)])

    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        """Embed the input and pass it through the stacked decoder layers.

        Args:
            x: Input token ids of shape (batch_size, seq_len).
            mask: Optional attention mask, applied only to the first layer.

        Returns:
            Output tensor of shape (batch_size, seq_len, hidden_size).
        """
        x = self.embeddings(x)
        x = self.layers[0](x, mask)      # enough to apply mask to just the first layer
        for layer in self.layers[1: ]:
            x = layer(x)
        return x

### Sanity check

Instantiate the decoder from `config` and run a forward pass on the tokenized input. The output shape `(batch_size, seq_len, hidden_size)` confirms the whole stack — embeddings, masked multi-head attention, and feed-forward layers — runs end to end.

In [66]:
decoder = TransformerDecoder(config)
decoder(inputs.input_ids, mask).size()

torch.Size([1, 5, 768])

In [67]:
decoder

TransformerDecoder(
  (embeddings): Embeddings(
    (token_embeddings): Embedding(30522, 768)
    (position_embeddings): Embedding(512, 768)
    (layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.5, inplace=False)
  )
  (layers): ModuleList(
    (0-11): 12 x TransformerDecoderLayer(
      (layer_norm_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
      (layer_norm_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
      (attention): MultiHeadAttention(
        (heads): ModuleList(
          (0-11): 12 x AttentionHead(
            (q): Linear(in_features=768, out_features=64, bias=True)
            (k): Linear(in_features=768, out_features=64, bias=True)
            (v): Linear(in_features=768, out_features=64, bias=True)
          )
        )
        (output_linear): Linear(in_features=768, out_features=768, bias=True)
      )
      (feed_forward): FeedForward(
        (linear_1): Linear(in_f